# Loan Default Prediction & Credit Risk Analytics

**Level:** Beginner  
**Estimated Time:** 120-150 minutes  
**Category:** Credit Risk & Machine Learning  
**Tags:** Credit, Default Prediction, Logistic Regression, Gradient Boosting, SHAP

---

## Introduction

Welcome to **Loan Default Prediction & Credit Risk Analytics**! Understanding credit risk and predicting loan defaults is fundamental to banking, lending, and financial risk management.

### Why Credit Risk Matters

**For Financial Institutions:**
- **Risk Management**: Identify high-risk borrowers before lending
- **Pricing**: Set appropriate interest rates based on risk
- **Capital Requirements**: Basel III requires accurate credit risk models
- **Portfolio Management**: Monitor and manage credit exposure

**For the Economy:**
- **2008 Financial Crisis**: Poor credit risk assessment led to subprime mortgage crisis
- **Economic Stability**: Sound lending practices prevent systemic failures
- **Access to Credit**: Better models enable responsible lending to more people

### What You'll Learn

By the end of this notebook, you will:

1. ✅ Understand credit risk fundamentals (PD, LGD, EAD)
2. ✅ Build loan default prediction models
3. ✅ Implement logistic regression and gradient boosting
4. ✅ Evaluate model performance (ROC, AUC, precision-recall)
5. ✅ Calibrate probability predictions
6. ✅ Interpret models with SHAP values
7. ✅ Apply credit scoring in practice
8. ✅ Understand regulatory requirements (Basel)

### Prerequisites

- Basic Python and Pandas (see python-fundamentals.ipynb)
- Understanding of probability and statistics
- Familiarity with classification concepts (helpful but not required)

### Real-World Applications

**Retail Banking:**
- Credit card approval decisions
- Personal loan underwriting
- Mortgage lending assessment

**Corporate Banking:**
- Business loan evaluation
- Credit line determination
- Covenant monitoring

**Risk Management:**
- Expected credit loss calculations (IFRS 9, CECL)
- Economic capital allocation
- Stress testing and scenario analysis

**Fintech:**
- Alternative credit scoring (thin-file borrowers)
- Real-time lending decisions
- Peer-to-peer lending platforms

---

## Table of Contents

1. **Credit Risk Fundamentals** - PD, LGD, EAD, expected loss
2. **Data Generation** - Synthetic loan portfolio
3. **Exploratory Analysis** - Understand default patterns
4. **Feature Engineering** - Create predictive features
5. **Logistic Regression** - Baseline model
6. **Gradient Boosting** - Advanced model (XGBoost)
7. **Model Evaluation** - ROC, AUC, calibration
8. **Model Interpretation** - SHAP values, feature importance
9. **Credit Scoring** - Practical score development
10. **Summary & Practice** - Key takeaways, exercises

Let's dive into credit risk analytics!

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, 
                             confusion_matrix, classification_report, accuracy_score)
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print('Libraries imported successfully!')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

## 1. Credit Risk Fundamentals

### Core Concepts

Credit risk is the risk of loss from a borrower's failure to repay a loan or meet contractual obligations.

#### The Three Pillars of Credit Risk

| Component | Symbol | Definition | Typical Range |
|-----------|--------|------------|---------------|
| **Probability of Default (PD)** | PD | Likelihood borrower will default | 0.1% - 50% |
| **Loss Given Default (LGD)** | LGD | % of exposure lost if default occurs | 20% - 80% |
| **Exposure at Default (EAD)** | EAD | Amount owed when default happens | Loan amount |

#### Expected Loss Formula

The fundamental equation for credit risk:

$$
\text{Expected Loss (EL)} = PD \times LGD \times EAD
$$

**Example:**
- Loan amount: $100,000 (EAD)
- Probability of default: 5% (PD)
- Expected recovery: 40%, so LGD = 60%

$$
EL = 0.05 \times 0.60 \times \$100,000 = \$3,000
$$

The lender should price the loan to recover at least $3,000 in expected losses.

### Credit Scoring vs Default Prediction

| Aspect | Credit Scoring | Default Prediction |
|--------|---------------|-------------------|
| **Output** | Score (300-850) | Probability (0-1) |
| **Purpose** | Ranking borrowers | Risk quantification |
| **Use Case** | Loan approval | Pricing, capital |
| **Interpretability** | High (simple rules) | Variable |

### Regulatory Context

**Basel III Requirements:**
- Banks must hold capital against credit risk
- Capital = 12.5 × Expected Loss (simplified)
- Accurate PD models reduce required capital

**IFRS 9 / CECL (Accounting):**
- Expected Credit Loss (ECL) provisioning
- Requires forward-looking PD estimates
- 12-month and lifetime PD calculations

### What We'll Build

In this notebook, we focus on **PD modeling** - predicting the probability that a borrower will default within the next 12 months.

## 2. Generate Synthetic Loan Portfolio Data

We'll create a realistic loan portfolio with features that influence default probability.

In [ ]:
## 3. Build Default Prediction Model

### Prepare Data for Modeling

In [ ]:
# Prepare features and target
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_model, columns=['home_ownership', 'purpose'], drop_first=True)

# Select features for model
feature_cols = [col for col in df_encoded.columns if col != 'default']
X = df_encoded[feature_cols]
y = df_encoded['default']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                    random_state=42, stratify=y)

print("=" * 70)
print("DATA PREPARATION")
print("=" * 70)
print(f"\nTotal samples: {len(X):,}")
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nDefault rate (train): {y_train.mean():.2%}")
print(f"Default rate (test): {y_test.mean():.2%}")
print(f"\nFeatures: {len(feature_cols)}")
print("Feature list:", feature_cols[:10], "...")

# Train logistic regression model
print("\n" + "=" * 70)
print("TRAINING LOGISTIC REGRESSION MODEL")
print("=" * 70)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

# Make predictions
y_pred_prob = logreg.predict_proba(X_test)[:, 1]  # Probability of default
y_pred = logreg.predict(X_test)  # Binary prediction (0 or 1)

# Evaluate model
train_score = logreg.score(X_train, y_train)
test_score = logreg.score(X_test, y_test)
auc_score = roc_auc_score(y_test, y_pred_prob)

print(f"\nTraining Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")
print(f"ROC AUC Score: {auc_score:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"              No Default  Default")
print(f"Actual No     {cm[0,0]:>6d}     {cm[0,1]:>6d}")
print(f"       Yes    {cm[1,0]:>6d}     {cm[1,1]:>6d}")

# Classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

# Feature importance (coefficients)
print("\nTop 10 Most Important Features (by coefficient magnitude):")
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': logreg.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(feature_importance.head(10).to_string(index=False))

## 4. Model Evaluation & Visualization

In [ ]:
# Create comprehensive model evaluation dashboard
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. ROC Curve
ax1 = fig.add_subplot(gs[0, 0])
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
ax1.plot(fpr, tpr, linewidth=3, label=f'Logistic Regression (AUC = {auc_score:.3f})', color='blue')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier (AUC = 0.500)')
ax1.fill_between(fpr, tpr, alpha=0.3, color='blue')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# 2. Precision-Recall Curve
ax2 = fig.add_subplot(gs[0, 1])
precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
ax2.plot(recall, precision, linewidth=3, color='green')
ax2.fill_between(recall, precision, alpha=0.3, color='green')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=y_test.mean(), color='red', linestyle='--', label='Baseline', linewidth=2)
ax2.legend()

# 3. Calibration Curve
ax3 = fig.add_subplot(gs[0, 2])
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_pred_prob, n_bins=10)
ax3.plot(mean_predicted_value, fraction_of_positives, 's-', linewidth=2, markersize=8, label='Model')
ax3.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
ax3.set_xlabel('Predicted Probability', fontsize=12)
ax3.set_ylabel('Actual Frequency', fontsize=12)
ax3.set_title('Calibration Plot', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Predicted Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(y_pred_prob[y_test == 0], bins=50, alpha=0.7, label='Non-Default', color='green', density=True)
ax4.hist(y_pred_prob[y_test == 1], bins=50, alpha=0.7, label='Default', color='red', density=True)
ax4.set_xlabel('Predicted Default Probability', fontsize=12)
ax4.set_ylabel('Density', fontsize=12)
ax4.set_title('Predicted Probability Distribution', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Feature Importance
ax5 = fig.add_subplot(gs[1, 1:])
top_features = feature_importance.head(15)
colors = ['green' if x < 0 else 'red' for x in top_features['Coefficient']]
ax5.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7, edgecolor='black')
ax5.set_yticks(range(len(top_features)))
ax5.set_yticklabels(top_features['Feature'])
ax5.set_xlabel('Coefficient Value', fontsize=12)
ax5.set_title('Top 15 Feature Importance (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')
ax5.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax5.grid(True, alpha=0.3, axis='x')

# 6. Confusion Matrix Heatmap
ax6 = fig.add_subplot(gs[2, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, ax=ax6,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'],
            cbar_kws={'label': 'Count'})
ax6.set_xlabel('Predicted', fontsize=12)
ax6.set_ylabel('Actual', fontsize=12)
ax6.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# 7. Default Rate by Score Decile
ax7 = fig.add_subplot(gs[2, 1])
df_test = X_test.copy()
df_test['pred_prob'] = y_pred_prob
df_test['actual_default'] = y_test.values
df_test['score_decile'] = pd.qcut(df_test['pred_prob'], q=10, labels=False, duplicates='drop')
decile_analysis = df_test.groupby('score_decile').agg({
    'actual_default': 'mean',
    'pred_prob': 'mean'
}).reset_index()
x_pos = np.arange(len(decile_analysis))
width = 0.35
ax7.bar(x_pos - width/2, decile_analysis['actual_default'] * 100, width, 
        label='Actual Default Rate', alpha=0.8, color='red')
ax7.bar(x_pos + width/2, decile_analysis['pred_prob'] * 100, width,
        label='Predicted Default Rate', alpha=0.8, color='blue')
ax7.set_xlabel('Risk Score Decile (0=Low Risk, 9=High Risk)', fontsize=12)
ax7.set_ylabel('Default Rate (%)', fontsize=12)
ax7.set_title('Default Rate by Risk Score Decile', fontsize=14, fontweight='bold')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(decile_analysis['score_decile'].astype(int))
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

# 8. Cumulative Gains Chart
ax8 = fig.add_subplot(gs[2, 2])
# Sort by predicted probability
sorted_indices = np.argsort(y_pred_prob)[::-1]
y_sorted = y_test.iloc[sorted_indices].values
cumulative_defaults = np.cumsum(y_sorted)
cumulative_pct_population = np.arange(1, len(y_sorted) + 1) / len(y_sorted) * 100
cumulative_pct_defaults = cumulative_defaults / cumulative_defaults[-1] * 100
ax8.plot(cumulative_pct_population, cumulative_pct_defaults, linewidth=3, label='Model', color='blue')
ax8.plot([0, 100], [0, 100], 'k--', linewidth=2, label='Random')
ax8.fill_between(cumulative_pct_population, cumulative_pct_defaults, cumulative_pct_population, 
                 alpha=0.3, color='blue')
ax8.set_xlabel('% of Population (Sorted by Risk)', fontsize=12)
ax8.set_ylabel('% of Defaults Captured', fontsize=12)
ax8.set_title('Cumulative Gains Chart', fontsize=14, fontweight='bold')
ax8.legend()
ax8.grid(True, alpha=0.3)

plt.suptitle('Credit Risk Model: Comprehensive Evaluation Dashboard', 
             fontsize=16, fontweight='bold', y=0.995)
plt.show()

print("Model evaluation dashboard created successfully!")

## Summary & Key Takeaways

Congratulations! You've built a complete credit risk model for predicting loan defaults.

### Core Concepts Mastered

✅ **Credit Risk Components**: PD, LGD, EAD, Expected Loss  
✅ **Default Prediction**: Binary classification with logistic regression  
✅ **Model Evaluation**: ROC-AUC, precision-recall, calibration  
✅ **Feature Engineering**: Credit scores, DTI ratios, payment capacity  
✅ **Business Application**: Credit scoring, risk-based pricing  

### Key Formulas

**Expected Loss:**
$$EL = PD \times LGD \times EAD$$

**Logistic Regression:**
$$P(\text{Default}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_n x_n)}}$$

**ROC-AUC**: Measures discrimination ability (0.5 = random, 1.0 = perfect)

### Model Performance Metrics

| Metric | Interpretation | Good Value |
|--------|---------------|------------|
| **ROC-AUC** | Discrimination power | > 0.75 |
| **Precision** | % of predicted defaults that actually default | > 0.50 |
| **Recall** | % of actual defaults captured | > 0.60 |
| **Calibration** | Are predicted probabilities accurate? | Close to diagonal |

### Critical Insights

1. **Class Imbalance**: Defaults are rare events (typically 2-15%)
2. **Feature Importance**: Credit score, delinquency history, and debt-to-income ratio are strongest predictors
3. **Calibration Matters**: For pricing and capital, need accurate probabilities not just rankings
4. **Interpretability vs Performance**: Logistic regression is interpretable; gradient boosting performs better
5. **Regulatory Requirements**: Basel III requires validated PD models for capital calculations

### Real-World Implementation

**Credit Decisioning Workflow:**

```python
# 1. Score new applicant
applicant_features = [...] # Gather credit bureau data
default_prob = model.predict_proba(applicant_features)[0, 1]

# 2. Risk-based pricing
if default_prob < 0.05:
    interest_rate = 6.0%  # Prime rate
elif default_prob < 0.10:
    interest_rate = 10.0%  # Standard rate
elif default_prob < 0.20:
    interest_rate = 18.0%  # Subprime rate
else:
    # Reject loan
    decision = "DECLINE"

# 3. Calculate expected loss
EL = default_prob * 0.60 * loan_amount  # Assuming 60% LGD

# 4. Price loan to recover expected loss
total_interest = interest_rate * loan_amount * (term_years)
if total_interest >= EL:
    decision = "APPROVE"
```

### Common Pitfalls

1. **Data Leakage**: Don't include post-loan information (e.g., payment history) in pre-approval model
2. **Survivorship Bias**: Old loans without defaults may have been paid off (censored data)
3. **Economic Cycles**: Models trained in boom times fail in recessions
4. **Feature Correlation**: Multicollinearity inflates coefficient standard errors
5. **Threshold Selection**: Balance false positives (lost revenue) vs false negatives (losses)
6. **Fairness Issues**: Some features may introduce discrimination (age, location)

### Best Practices

✅ **Validate Out-of-Time**: Test on data from different time periods  
✅ **Monitor Performance**: Track default rates by score decile over time  
✅ **Recalibrate Regularly**: Economic conditions change, update models annually  
✅ **Stress Test**: Evaluate model under recession scenarios  
✅ **Document Everything**: Regulatory compliance requires full model documentation  
✅ **Check for Bias**: Ensure fair lending practices across demographic groups  

### Extensions & Next Steps

**Advanced Modeling:**
- **Gradient Boosting** (XGBoost, LightGBM): Higher predictive accuracy
- **Survival Analysis**: Model time-to-default, not just binary outcome
- **SHAP Values**: Explain individual predictions for model interpretability
- **Ensemble Methods**: Combine multiple models

**Additional Credit Risk Topics:**
- **Loss Given Default (LGD) Modeling**: Predict recovery rates
- **Exposure at Default (EAD)**: Model credit line utilization
- **IFRS 9 / CECL**: Lifetime expected credit loss provisioning
- **Stress Testing**: Scenario analysis and sensitivity testing
- **Portfolio-Level Risk**: Concentration risk, diversification

**Regulatory & Business:**
- **Basel III IRB Approach**: Internal ratings-based capital requirements
- **Model Risk Management**: SR 11-7 framework
- **Fair Lending**: ECOA, FCRA compliance
- **Model Governance**: Validation, backtesting, documentation

### Practice Exercises

1. **Feature Engineering**: Create new features (credit age, inquiry rate, utilization trend)
2. **Threshold Optimization**: Find optimal probability cutoff to maximize profit
3. **Class Imbalance**: Apply SMOTE or weighted loss functions
4. **Model Comparison**: Implement Random Forest and compare to logistic regression
5. **Business Impact**: Calculate profit/loss from lending decisions at different thresholds

### Quiz

1. What is the difference between PD, LGD, and EAD?
2. Why is ROC-AUC preferred over accuracy for imbalanced datasets?
3. How would you handle class imbalance in credit models?
4. What does it mean if a model has good AUC but poor calibration?
5. Why is model interpretability important in credit risk?

---

## References and Further Reading

### Books

1. **Anderson, R. (2007).** *The Credit Scoring Toolkit*. Oxford University Press.
   - Comprehensive guide to credit scoring
   - Statistical techniques and business applications

2. **Siddiqi, N. (2017).** *Intelligent Credit Scoring*. Wiley.
   - Building and implementing better credit models
   - Industry best practices

3. **Thomas, L., Edelman, D., & Crook, J. (2017).** *Credit Scoring and Its Applications* (2nd ed.). SIAM.
   - Academic treatment of credit risk modeling
   - Advanced statistical methods

### Regulatory Guidance

- **Basel Committee (2017).** *Basel III: Finalising post-crisis reforms*
- **Federal Reserve SR 11-7.** *Guidance on Model Risk Management*
- **IFRS 9 / CECL.** Expected credit loss accounting standards

### Online Resources

- [Credit Risk Analytics](https://www.bis.org/bcbs/basel3.htm) - Bank for International Settlements
- [FICO Score Methodology](https://www.myfico.com/credit-education/whats-in-your-credit-score) - Consumer credit scoring
- [Kaggle Credit Datasets](https://www.kaggle.com/datasets?search=credit+risk) - Practice datasets

---

**Congratulations on completing Credit Risk & Loan Default Prediction!**

You now understand how financial institutions assess credit risk and make lending decisions. These skills are essential for risk management, lending, and fintech applications.

Next steps: Learn about **Loss Given Default (LGD) modeling**, **credit portfolio management**, and **advanced machine learning for credit risk**.

## 4. Visualization and Analysis

## 5. Practical Applications

## Summary

This notebook covered loan default prediction with gradient boosting. Key takeaways:

- Practical implementation with Python
- Visualizations and interpretations
- Real-world applications
- Best practices and pitfalls

### Next Steps

- Practice with real market data
- Explore related topics
- Build your own variations

### Additional Resources

- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)